# 03. kb_chunks 청킹 (api_eflaw / api_prec / api_expc)

`data/01_raw/{api_eflaw,api_prec,api_expc}/`의 원본을 아래 `kb_chunks` (Supabase/pgvector) 스키마에 맞춰 청킹한다.

```sql
CREATE TABLE kb_chunks (
    id           bigserial PRIMARY KEY,
    source_type  text NOT NULL,   -- statute|precedent|interpretation|mediation_case|counsel_case|standard_contract|guide
    source_org   text,
    doc_title    text NOT NULL,
    doc_year     int,
    authority    text NOT NULL,   -- binding | persuasive | reference
    stage        text NOT NULL,   -- pre | post | both
    issue        text[] NOT NULL DEFAULT '{}',
    law_name     text,
    article      text,
    case_no      text,
    source_id    text NOT NULL,   -- 원본 고유키 (재적재 dedup용)
    chunk_index  int  NOT NULL DEFAULT 0,
    content      text NOT NULL,
    embedding    vector(1024) NOT NULL,
    UNIQUE (source_type, source_id, chunk_index)
);
```

| source_type | 원본 | 단위 |
|---|---|---|
| `statute` | `api_eflaw/*.json` (법령 16개) | 조문(제N조) 1개 = 청크 1개 |
| `precedent` | `api_prec/prec_*.jsonl` | 사건 1건 → 500자/50자 오버랩 분할 |
| `interpretation` | `api_expc/expc_*.jsonl` | 안건 1건 → 500자/50자 오버랩 분할 |

임베딩·Supabase 적재는 이 노트북 범위 밖(별도 단계)이며, 여기서는 `data/04_chunks/kb_chunks.jsonl` 생성까지만 다룬다.

## 0. 공통 설정

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

RAW_DIR = Path("../data/01_raw")
EFLAW_DIR = RAW_DIR / "api_eflaw"
PREC_DIR = RAW_DIR / "api_prec"
EXPC_DIR = RAW_DIR / "api_expc"

CHUNKS_DIR = Path("../data/04_chunks")
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# 법령명(파일 stem) → (stage, issue[]) — 도메인 1차 판단값, 팀 검수 필요
LAW_STAGE_MAP: dict[str, tuple[str, list[str]]] = {
    "주택임대차보호법": ("both", ["갱신종료", "보증금권리", "경매배당"]),
    "주택임대차보호법_시행령": ("both", ["갱신종료", "보증금권리"]),
    "민법": ("both", []),
    "부동산등기법": ("pre", ["등기"]),
    "부동산등기규칙": ("pre", ["등기"]),
    "공인중개사법": ("pre", ["중개"]),
    "공인중개사법_시행규칙": ("pre", ["중개"]),
    "민사집행법": ("post", ["경매배당"]),
    "국세징수법": ("post", ["경매배당"]),
    "전세사기피해자_지원_및_주거안정에_관한_특별법": ("pre", ["전세사기"]),
    "부동산_거래신고_등에_관한_법률": ("pre", ["신고"]),
    "상가건물_임대차보호법": ("both", ["상가임대차"]),
    "주민등록법": ("pre", ["대항력"]),
    "민간임대주택에_관한_특별법": ("both", []),
    "집합건물의_소유_및_관리에_관한_법률": ("post", []),
    "주택도시기금법": ("pre", []),
}

# 판례/해석례 _category(파일명 유래 토픽) → stage — prec/expc 공용
TOPIC_STAGE_MAP: dict[str, str] = {
    "전세사기": "pre",
    "중개": "pre",
    "갱신종료": "post",
    "경매배당": "post",
    "수선원상회복": "post",
    "보증금권리": "both",
    "상가임대차": "both",
}

SPLIT_CHUNK_SIZE = 500
SPLIT_CHUNK_OVERLAP = 50
_splitter = RecursiveCharacterTextSplitter(chunk_size=SPLIT_CHUNK_SIZE, chunk_overlap=SPLIT_CHUNK_OVERLAP)

print("EFLAW_DIR:", EFLAW_DIR.resolve())
print("PREC_DIR :", PREC_DIR.resolve())
print("EXPC_DIR :", EXPC_DIR.resolve())
print("CHUNKS_DIR:", CHUNKS_DIR.resolve())

### 공통 유틸

In [ ]:
def _as_list(x):
    """law.go.kr API는 원소 1개면 dict, 여러 개면 list로 온다 — 항상 list로 정규화."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


_REVISION_NOTE_RE = re.compile(r"<[^>]{0,30}(?:개정|신설|삭제|전문개정|일부개정)[^>]*>")


def _as_text(x) -> str:
    """드물게 내용 필드가 list[str]로 오는 경우(예: 긴 항내용)가 있어 문자열로 정규화."""
    if isinstance(x, list):
        return "\n".join(_as_text(i) for i in x)
    return x or ""


def _strip_revision_notes(text) -> str:
    """조문 본문에 섞인 '<개정 2020.6.9>' 류 이력 주석 제거 (임베딩 노이즈)."""
    text = _as_text(text)
    if not text:
        return ""
    return _REVISION_NOTE_RE.sub("", text).strip()


def _clean_html(text) -> str:
    """<br/> → 개행, 나머지 태그 제거, 공백 정리 (판례내용 등 HTML형 텍스트용)."""
    text = _as_text(text)
    if not text:
        return ""
    text = re.sub(r"<br\s*/?>", "\n", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def _year_of(date_str) -> int | None:
    s = str(date_str or "")
    return int(s[:4]) if s[:4].isdigit() else None


def _iter_jsonl(path: Path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

## 1. statute (법령 조문)

In [ ]:
def _build_article_text(article: dict) -> str:
    """조문내용 + 항(항내용) + 호(호내용)를 원문 순서대로 이어붙인다."""
    parts = [_strip_revision_notes(article.get("조문내용", ""))]
    for hang in _as_list(article.get("항")):
        hang_text = _strip_revision_notes(hang.get("항내용", ""))
        if hang_text:
            parts.append(hang_text)
        for ho in _as_list(hang.get("호")):
            ho_text = _strip_revision_notes(ho.get("호내용", ""))
            if ho_text:
                parts.append(ho_text)
    for ho in _as_list(article.get("호")):  # 항 없이 조문에 바로 호가 붙는 경우
        ho_text = _strip_revision_notes(ho.get("호내용", ""))
        if ho_text:
            parts.append(ho_text)
    return "\n".join(p for p in parts if p).strip()


def build_statute_chunks(path: Path) -> list[dict]:
    raw = json.loads(path.read_text(encoding="utf-8"))

    law_name = raw.get("법령명한글", "")
    law_id = raw.get("법령ID", "")
    source_org = raw.get("소관부처명")
    doc_year = _year_of(raw.get("시행일자"))
    stage, issue = LAW_STAGE_MAP.get(path.stem, ("both", []))

    articles = _as_list(raw.get("본문", {}).get("법령", {}).get("조문", {}).get("조문단위"))

    chunks = []
    for art in articles:
        if art.get("조문여부") != "조문":
            continue  # "전문"(장/절 표제)은 실제 조문이 아니므로 제외

        content = _build_article_text(art)
        if not content:
            continue

        no = art.get("조문번호", "")
        branch = art.get("조문가지번호", "")
        article_no = f"{no}의{branch}" if branch and branch not in ("0", "00") else no

        chunks.append({
            "source_type": "statute",
            "source_org": source_org,
            "doc_title": law_name,
            "doc_year": doc_year,
            "authority": "binding",
            "stage": stage,
            "issue": issue,
            "law_name": law_name,
            "article": article_no,
            "case_no": None,
            "source_id": f"{law_id}_{art.get('조문키', '')}",
            "chunk_index": 0,
            "content": content,
        })
    return chunks

## 2. precedent (판례)

In [ ]:
_REASON_MARKER_RE = re.compile(r"【\s*이\s*유\s*】")


def build_precedent_chunks(path: Path) -> list[dict]:
    chunks = []
    for rec in _iter_jsonl(path):
        body = rec.get("본문", {})
        body = body.get("PrecService", body) if isinstance(body, dict) else {}

        판시사항 = _clean_html(body.get("판시사항", ""))
        판결요지 = _clean_html(body.get("판결요지", ""))
        판례내용 = body.get("판례내용", "") or ""
        m = _REASON_MARKER_RE.search(판례내용)
        if m:
            판례내용 = 판례내용[m.end():]
        판례내용 = _clean_html(판례내용)

        full_text = "\n\n".join(p for p in [판시사항, 판결요지, 판례내용] if p)
        if not full_text:
            continue

        category = rec.get("_category", "")
        stage = TOPIC_STAGE_MAP.get(category, "both")
        court = rec.get("법원명", "")
        authority = "binding" if "대법원" in court else "persuasive"
        source_id = str(rec.get("판례일련번호", ""))

        for i, text in enumerate(_splitter.split_text(full_text)):
            chunks.append({
                "source_type": "precedent",
                "source_org": court,
                "doc_title": rec.get("사건명", ""),
                "doc_year": _year_of(rec.get("선고일자", "")),
                "authority": authority,
                "stage": stage,
                "issue": [category] if category else [],
                "law_name": None,
                "article": None,
                "case_no": rec.get("사건번호", ""),
                "source_id": source_id,
                "chunk_index": i,
                "content": text,
            })
    return chunks

## 3. interpretation (법령해석례)

In [ ]:
_BRACKET_LAW_RE = re.compile(r"「([^」]+)」")
_ARTICLE_RE = re.compile(r"제\d+조(?:의\d+)?")


def build_interpretation_chunks(path: Path) -> list[dict]:
    chunks = []
    for rec in _iter_jsonl(path):
        body = rec.get("본문", {})
        body = body.get("ExpcService", body) if isinstance(body, dict) else {}

        질의요지 = _clean_html(body.get("질의요지", ""))
        회답 = _clean_html(body.get("회답", ""))
        이유 = _clean_html(body.get("이유", ""))

        parts = []
        if 질의요지:
            parts.append(f"질의요지: {질의요지}")
        if 회답:
            parts.append(f"회답: {회답}")
        if 이유:
            parts.append(f"이유: {이유}")
        full_text = "\n\n".join(parts)
        if not full_text:
            continue

        category = rec.get("_category", "")
        stage = TOPIC_STAGE_MAP.get(category, "both")
        안건명 = rec.get("안건명", "")
        law_match = _BRACKET_LAW_RE.search(안건명)
        law_name = law_match.group(1) if law_match else None
        article_matches = list(dict.fromkeys(_ARTICLE_RE.findall(안건명)))
        article = ", ".join(article_matches) if article_matches else None
        source_id = str(rec.get("법령해석례일련번호", ""))

        for i, text in enumerate(_splitter.split_text(full_text)):
            chunks.append({
                "source_type": "interpretation",
                "source_org": rec.get("회신기관명", ""),
                "doc_title": 안건명,
                "doc_year": _year_of(rec.get("회신일자", "")),
                "authority": "persuasive",
                "stage": stage,
                "issue": [category] if category else [],
                "law_name": law_name,
                "article": article,
                "case_no": rec.get("안건번호", ""),
                "source_id": source_id,
                "chunk_index": i,
                "content": text,
            })
    return chunks

## 4. 전체 실행 + 검증

In [ ]:
all_chunks: list[dict] = []

for p in sorted(EFLAW_DIR.glob("*.json")):
    n = build_statute_chunks(p)
    print(f"[statute] {p.stem}: {len(n)}개 조문")
    all_chunks.extend(n)

for p in sorted(PREC_DIR.glob("prec_*.jsonl")):
    if p.stem == "prec_checkpoint":
        continue  # 재수집용 체크포인트 — 카테고리 파일과 내용이 중복되므로 제외
    n = build_precedent_chunks(p)
    print(f"[precedent] {p.stem}: {len(n)}개 청크")
    all_chunks.extend(n)

for p in sorted(EXPC_DIR.glob("expc_*.jsonl")):
    n = build_interpretation_chunks(p)
    print(f"[interpretation] {p.stem}: {len(n)}개 청크")
    all_chunks.extend(n)

print(f"\n총 청크 수: {len(all_chunks)}")

In [ ]:
df = pd.DataFrame(all_chunks)

print("source_type별 건수\n", df["source_type"].value_counts(), "\n")
print("stage별 건수\n", df["stage"].value_counts(), "\n")
print("authority별 건수\n", df["authority"].value_counts(), "\n")

empty_content = (df["content"].str.strip() == "").sum()
print("빈 content 건수:", empty_content)

dup_key = df.duplicated(subset=["source_type", "source_id", "chunk_index"]).sum()
print("(source_type, source_id, chunk_index) 중복 건수:", dup_key)

df[df["source_type"] == "statute"].sample(1)[["doc_title", "article", "stage", "issue", "content"]]

In [ ]:
out_path = CHUNKS_DIR / "kb_chunks.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for rec in all_chunks:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"저장 완료: {len(all_chunks)}건 → {out_path.resolve()}")